In [8]:
import pathlib
import ollama
from itertools import product
import re

In [9]:
OUTPUT = '../data/output.txt'
PROMPTS = '../prompts/prompts.txt'

In [10]:
text = []

system_prompt = "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze. Timestamp ist immer in Sekunden. Jedes Attribut hat zwei Werte: Das Erste steht immer für die zu bewertende Runde, der zweite Wert beschreibt eine optimale Runde die dir als Referenz dient. **Verwende dafür ausschließlich die Daten aus der Liste des Users**. Negative Prompt: Denk dir keine weiteren Daten aus, Bewerte nicht die zweiten Werte der Attribute"
user_prompt = "Bewerte meine Fahrleistung, zeige mir klar die Unterschiede:"

with open(OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if line != '\'':
            text.append(line + '\n')

# list(product([0], [0, 1, 2, 3, 4, 5])) + list(product([1,2], [0,1,2,3,4]))
for lap, segment in list(product([0], [0, 1, 2, 3, 4])):
    print(f'Lap: {lap}, Segment: {segment}')
    
    # Filter text for lap and segment
    filterd_lines = [row for row in text if f"'lap_number': {lap}" in row and f"'segment': {segment}" in row]
    filtered_text = '\n'.join(filterd_lines)
    
    # Generate LLM text from output file
    resp = ollama.chat(
        model="nemotron-3-nano:30b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```json\n{filtered_text}\n```"},
        ],
    )

    # Get min and max timestamps from filtered_lines for logging
    min_timestamp, max_timestamp = float('inf'), float('-inf')
    for line in filterd_lines:
        num = r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?"
        pattern = rf"timestamp'\s*:\s*({num})\s*,\s*({num})"
        timestamp = float(re.search(pattern, line).group(1))

        min_timestamp = min(min_timestamp, timestamp)
        max_timestamp = max(max_timestamp, timestamp)

     # Log prompts and responses
    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            "System Prompt: " + str(system_prompt) + "\n",
            "User Prompt: " + str(user_prompt) + "\n",
            "Response: " + str(resp["message"]["content"]) + "\n\n",
        ])

Lap: 0, Segment: 0
Lap: 0, Segment: 1
Lap: 0, Segment: 2
Lap: 0, Segment: 3
Lap: 0, Segment: 4
